# Virtual concatenation of embedding chunks

Uses [h5py Virtual Datasets](https://docs.h5py.org/en/stable/vds.html) to concatenate
chunk `.h5ad` files **without copying any data**. The result is a small `.h5` file
whose `obsm/X_state` transparently reads from the original chunks on the fly.

Inspired by [scverse/anndata#2032](https://github.com/scverse/anndata/pull/2032).

In [2]:
from pathlib import Path
import re

import h5py
import numpy as np
import pandas as pd
import anndata as ad
from tqdm.auto import tqdm

# ===== User config =====
INPUT_H5AD = Path("/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/tahoe.h5ad")
CHUNKS_DIR = Path("/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/tahoe/test")
VIRTUAL_OUT = Path("/lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/tahoe/tahoe_se_virtual.h5")
OBSM_KEY = "X_state"

/home/icb/xiaotong.fu/miniconda3/envs/pancellflow/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Scan chunk shapes

In [3]:
def chunk_sort_key(path: Path):
    nums = re.findall(r"\d+", path.stem)
    return tuple(int(x) for x in nums) if nums else (path.name,)

chunk_files = sorted(CHUNKS_DIR.glob("*.h5ad"), key=chunk_sort_key)
print(f"Found {len(chunk_files)} chunk files")

chunk_shapes = []
for fp in tqdm(chunk_files, desc="Scanning shapes"):
    with h5py.File(fp, "r") as f:
        chunk_shapes.append(f[f"obsm/{OBSM_KEY}"].shape)

total_rows = sum(s[0] for s in chunk_shapes)
emb_dim = chunk_shapes[0][1]
assert all(s[1] == emb_dim for s in chunk_shapes)
print(f"Total rows: {total_rows:,}  |  Embedding dim: {emb_dim}")

Found 200 chunk files


Scanning shapes:   0%|          | 0/200 [00:00<?, ?it/s]

Scanning shapes: 100%|██████████| 200/200 [00:23<00:00,  8.36it/s]

Total rows: 89,423,257  |  Embedding dim: 2058


## 2. Build virtual dataset

In [4]:
layout = h5py.VirtualLayout(shape=(total_rows, emb_dim), dtype=np.float32)

offset = 0
for fp, shape in tqdm(zip(chunk_files, chunk_shapes), total=len(chunk_files),
                      desc="Mapping sources"):
    n = shape[0]
    vsource = h5py.VirtualSource(
        str(fp), f"obsm/{OBSM_KEY}", shape=shape, dtype=np.float32,
    )
    layout[offset : offset + n] = vsource
    offset += n

VIRTUAL_OUT.parent.mkdir(parents=True, exist_ok=True)
with h5py.File(VIRTUAL_OUT, "w") as f:
    f.create_virtual_dataset(f"obsm/{OBSM_KEY}", layout)

print(f"Wrote virtual file: {VIRTUAL_OUT}")
print(f"File size: {VIRTUAL_OUT.stat().st_size / 1024:.1f} KB (metadata only)")

Mapping sources: 100%|██████████| 200/200 [00:00<00:00, 11609.24it/s]

Wrote virtual file: /lustre/groups/ml01/workspace/xiaotong.fu/data/pancellflow/tahoe/tahoe_se_virtual.h5
File size: 32.9 KB (metadata only)


## 3. Sanity check: read back & verify row count

In [5]:
with h5py.File(VIRTUAL_OUT, "r") as f:
    vds = f[f"obsm/{OBSM_KEY}"]
    print(f"Virtual dataset shape: {vds.shape}, dtype: {vds.dtype}")
    assert vds.shape == (total_rows, emb_dim)
    # spot-check: read first and last row
    print(f"First row norm:  {np.linalg.norm(vds[0]):.4f}")
    print(f"Last row norm:   {np.linalg.norm(vds[-1]):.4f}")

Virtual dataset shape: (89423257, 2058), dtype: float32
First row norm:  1.1960
Last row norm:   1.3248


## 4. Load obs from original input

In [6]:
adata_in = ad.read_h5ad(INPUT_H5AD, backed="r")
obs = adata_in.obs.copy()
del adata_in

assert len(obs) == total_rows, (
    f"Row mismatch: obs has {len(obs):,} rows but virtual dataset has {total_rows:,}"
)
print(f"obs loaded: {len(obs):,} rows, {len(obs.columns)} columns")
print(f"Cell lines: {obs['cell_line'].nunique()} unique")
obs.head()

obs loaded: 89,423,257 rows, 44 columns
Cell lines: 50 unique


,sample,species,gene_count,tscp_count,mread_count,bc1_wind,bc2_wind,bc3_wind,bc1_well,bc2_well,...,BARCODE,pcnt_mito,S_score,G2M_score,phase,cell_line_orig,pass_filter,cell_name,dosage,plate
01_001_025-lib_841,smp_1495,hg38,1676,2441,2892,1,1,25,A1,A1,...,01_001_025,0.025399,-0.066667,-0.095055,G1,CVCL_0131,full,A-172,0.05,plate_1
01_001_026-lib_841,smp_1495,hg38,1657,2454,2925,1,1,26,A1,A1,...,01_001_026,0.042787,0.128571,0.650549,G2M,CVCL_0480,full,PANC-1,0.05,plate_1
01_001_048-lib_841,smp_1495,hg38,1749,2521,2963,1,1,48,A1,A1,...,01_001_048,0.056724,0.242857,0.308791,G2M,CVCL_0293,full,HEC-1-A,0.05,plate_1
01_001_076-lib_841,smp_1495,hg38,834,1038,1258,1,1,76,A1,A1,...,01_001_076,0.066474,0.009524,0.245788,G2M,CVCL_0397,full,LS 180,0.05,plate_1
01_001_088-lib_841,smp_1495,hg38,1275,1710,2006,1,1,88,A1,A1,...,01_001_088,0.028655,-0.100000,-0.085348,G1,CVCL_1097,full,C32,0.05,plate_1


## 5. Query: retrieve embeddings for a specific cell line

In [7]:
EMB_CHUNK = 100_000

def get_embeddings_for(obs_df: pd.DataFrame, virtual_h5: Path, obsm_key: str,
                       **filters) -> tuple[pd.DataFrame, np.ndarray]:
    """
    Filter obs by column values and lazily fetch only the matching embeddings.

    Usage:
        sub_obs, sub_emb = get_embeddings_for(obs, VIRTUAL_OUT, OBSM_KEY,
                                              cell_line="A549", drug="Imatinib")
    """
    mask = pd.Series(True, index=obs_df.index)
    for col, val in filters.items():
        if isinstance(val, (list, tuple, set)):
            mask &= obs_df[col].isin(val)
        else:
            mask &= obs_df[col] == val

    sub_obs = obs_df.loc[mask]
    idx = np.where(mask.values)[0]
    n = len(idx)
    print(f"Filter {filters} -> {n:,} cells")

    with h5py.File(virtual_h5, "r") as f:
        vds = f[f"obsm/{obsm_key}"]
        emb_dim = vds.shape[1]
        sort_order = np.argsort(idx)
        sorted_idx = idx[sort_order]

        emb_sorted = np.empty((n, emb_dim), dtype=np.float32)
        for start in tqdm(range(0, n, EMB_CHUNK),
                          total=(n + EMB_CHUNK - 1) // EMB_CHUNK,
                          desc="Reading embeddings"):
            end = min(start + EMB_CHUNK, n)
            emb_sorted[start:end] = vds[sorted_idx[start:end].tolist()]

        emb = np.empty_like(emb_sorted)
        emb[sort_order] = emb_sorted

    return sub_obs, emb

In [8]:
# Show available cell lines
print(obs["cell_line"].value_counts())

cell_line
CVCL_0546    5655803
CVCL_0459    5371892
CVCL_0480    3879525
CVCL_1285    3097799
CVCL_0399    2766642
CVCL_1056    2570147
CVCL_0334    2463846
CVCL_0023    2392623
CVCL_0428    2364954
CVCL_0293    2296542
CVCL_1119    2266999
CVCL_1693    2208369
CVCL_0359    2193945
CVCL_0131    2123892
CVCL_1550    2091040
CVCL_1381    2087841
CVCL_0152    2084965
CVCL_0371    2070435
CVCL_0504    2056209
CVCL_0320    2014539
CVCL_1478    1867186
CVCL_1097    1836755
CVCL_1717    1779390
CVCL_1517    1750997
CVCL_1495    1729086
CVCL_0397    1690106
CVCL_1547    1643941
CVCL_0332    1628017
CVCL_1731    1594891
CVCL_1635    1591090
CVCL_C466    1555324
CVCL_1666    1512779
CVCL_0069    1462604
CVCL_0292    1413610
CVCL_1094    1398407
CVCL_0366    1377826
CVCL_0218    1346010
CVCL_1055    1344815
CVCL_0099    1172934
CVCL_0179    1170141
CVCL_1098    1115086
CVCL_1239     980228
CVCL_1724     874364
CVCL_0028     528281
CVCL_1125     510288
CVCL_1716     192280
CVCL_1715     133073
CVC

In [ ]:
# Example: get embeddings for a specific cell line
QUERY_CELL_LINE = obs["cell_line"].value_counts().index[0]  # most common

sub_obs, sub_emb = get_embeddings_for(
    obs, VIRTUAL_OUT, OBSM_KEY,
    cell_line=QUERY_CELL_LINE,
)
print(f"Cell line: {QUERY_CELL_LINE}")
print(f"Embeddings shape: {sub_emb.shape}")
print(f"Drugs in this cell line: {sub_obs['drug'].nunique()}")
sub_obs.head()

Filter {'cell_line': 'CVCL_0546'} -> 5,655,803 cells


In [ ]:
# Example: filter by cell line AND drug
sub_obs2, sub_emb2 = get_embeddings_for(
    obs, VIRTUAL_OUT, OBSM_KEY,
    cell_line=QUERY_CELL_LINE,
    drug=sub_obs["drug"].value_counts().index[0],
)
print(f"Embeddings shape: {sub_emb2.shape}")
sub_obs2.head()